# 05 Final Load Prep

This notebook finalizes our dataset by combining spending patterns with loan risk models, directly addressing our problem statement to predict default risks.

In [1]:
import pandas as pd
import os

data_path = '../data/processed/'

# Load all 4 CSVs
dim_customers = pd.read_csv(os.path.join(data_path, 'dim_customers_accounts.csv'))
fact_tx = pd.read_csv(os.path.join(data_path, 'fact_transactions.csv'))
fact_loans = pd.read_csv(os.path.join(data_path, 'fact_loans.csv'))
tableau_ready_dataset = pd.read_csv(os.path.join(data_path, 'tableau_ready_dataset.csv'))

## 1. Create the Ultimate Risk-Scored Master Dataset
We will take the tableau dataset and enrich it with Loan Status to allow Tableau to filter spending by 'Overdue' vs 'Active' customers.

In [2]:
# Merge Loan Status into the main tableau dataset based on AccountID
loan_simplified = fact_loans[['AccountID', 'StatusName', 'PrincipalAmount']]
loan_simplified.rename(columns={'StatusName': 'Loan_Status'}, inplace=True)

# Remove duplicates in case one account has multiple loans (take the worst status: Overdue > Active > Paid Off)
loan_simplified['Status_Rank'] = loan_simplified['Loan_Status'].map({'Overdue': 1, 'Active': 2, 'Paid Off': 3})
loan_simplified = loan_simplified.sort_values('Status_Rank').drop_duplicates(subset=['AccountID'], keep='first')
loan_simplified.drop(columns=['Status_Rank'], inplace=True)

# Merge
final_tableau_df = tableau_ready_dataset.merge(loan_simplified, on='AccountID', how='left')
final_tableau_df['Loan_Status'] = final_tableau_df['Loan_Status'].fillna('No Loan')
final_tableau_df['PrincipalAmount'] = final_tableau_df['PrincipalAmount'].fillna(0)

display(final_tableau_df.head())

/var/folders/8g/_6xbb_gs2w9b7qsq4txkkhs00000gn/T/ipykernel_66090/938161173.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  loan_simplified.rename(columns={'StatusName': 'Loan_Status'}, inplace=True)
/var/folders/8g/_6xbb_gs2w9b7qsq4txkkhs00000gn/T/ipykernel_66090/938161173.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  loan_simplified['Status_Rank'] = loan_simplified['Loan_Status'].map({'Overdue': 1, 'Active': 2, 'Paid Off': 3})


,TransactionID,AccountOriginID,AccountDestinationID,TransactionTypeID,Amount,TransactionDate,BranchID,Description,AccountID,CustomerID,...,AccountStatusID,Balance,OpeningDate,FirstName,LastName,DateOfBirth,AddressID,CustomerTypeID,Loan_Status,PrincipalAmount
0,3022681,201164,200868,2,855.17,2023-04-20 02:00:00.000000,41,Transaction 22681,201164,10426,...,1,55889.89,2018-11-14 00:00:00.000000,Damion,Compton,1991-07-27 00:00:00.000000,1190,1,Active,72030.09
1,3037846,200138,201402,2,806.20,2021-08-10 15:00:00.000000,43,Transaction 37846,200138,10735,...,3,35239.90,2019-03-04 00:00:00.000000,Jonas,Santiago,1996-01-06 00:00:00.000000,1015,3,No Loan,0.00
2,3045293,201002,201180,1,1229.44,2020-08-16 03:00:00.000000,5,Transaction 45293,201002,10458,...,1,92795.90,2020-09-20 00:00:00.000000,Belkis,Trevino,1994-05-13 00:00:00.000000,832,2,No Loan,0.00
3,3017397,201066,201144,4,4441.60,2021-10-10 06:00:00.000000,14,Transaction 17397,201066,10557,...,1,48854.26,2020-12-30 00:00:00.000000,Loyd,Baird,1963-02-04 00:00:00.000000,47,2,No Loan,0.00
4,3016750,200289,201413,3,2526.20,2022-07-28 00:00:00.000000,37,Transaction 16750,200289,10699,...,1,59032.48,2018-11-13 00:00:00.000000,Claudio,NaN,1983-10-15 00:00:00.000000,614,2,No Loan,0.00


## 2. Export Final File for Tableau Dashboard

In [3]:
# Save Final Tableau-ready dataset
tableau_path = os.path.join(data_path, 'final_risk_tableau_dataset.csv')
final_tableau_df.to_csv(tableau_path, index=False)
print(f"Saved the ultimate risk-enriched dataset to {tableau_path} for Dashboarding.")

Saved the ultimate risk-enriched dataset to ../data/processed/final_risk_tableau_dataset.csv for Dashboarding.
